In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
valuation_date = '2025-11-14'
db_path        = 'arms_database.db'
threshold      = 0.50   # only show bonds where |price change| > this

from datetime import datetime as _dt
_vd = _dt.strptime(valuation_date, '%Y-%m-%d')
output_file_path = rf'P:\Application\Risk Mgmt\MRM\ARMS\output\Price Attribution {valuation_date}.xlsx'

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.graph_objects as go
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Previous business day ─────────────────────────────────────────────────────
def prev_bday(date_str):
    d = datetime.strptime(date_str, '%Y-%m-%d') - timedelta(days=1)
    while d.weekday() >= 5:
        d -= timedelta(days=1)
    return d.strftime('%Y-%m-%d')

prev_date = prev_bday(valuation_date)
print(f'Period:  {prev_date}  →  {valuation_date}')

In [ ]:
# ── Load data from database ───────────────────────────────────────────────────
with sqlite3.connect(f'file:{db_path}?mode=ro', uri=True, timeout=10) as conn:

    # Bond metrics (both dates)
    df_m = pd.read_sql_query(f"""
        SELECT RepDate, isin, metric, value FROM bond_metrics
        WHERE RepDate IN ('{prev_date}', '{valuation_date}')
          AND metric IN ('clean_price_hybrid','mod_duration','dv01','cr01',
                         'par_yield_rf_rate_pp','dtm','ytm_dec')
    """, conn)

    # Bond prices / OAS (both dates)
    df_p = pd.read_sql_query(f"""
        SELECT RepDate, isin, oas_spread, ytm_bid, pricing_source
        FROM bond_price
        WHERE RepDate IN ('{prev_date}', '{valuation_date}')
    """, conn)

    # Bond dictionary
    df_bonds = pd.read_sql_query("""
        SELECT db.isin, db.bond_name, db.class_internal, db.coupon,
               db.maturity, db.coupon_type, db.par_value,
               di.company_name AS issuer
        FROM dic_bonds db
        LEFT JOIN dic_issuers di ON db.bloom_company_id = di.bloom_company_id
    """, conn)

    # Positions (portfolio type)
    df_pos = pd.read_sql_query(f"""
        SELECT isin, portfolio_type, currency,
               SUM(quantity) AS quantity,
               SUM(total_notional) AS face_value
        FROM positions
        WHERE RepDate = '{valuation_date}'
        GROUP BY isin, portfolio_type, currency
    """, conn)

    # Coupon / principal payments in this period
    df_cf = pd.read_sql_query(f"""
        SELECT isin, cashflows_date, coupon, principal
        FROM dic_bond_cf
        WHERE cashflows_date > '{prev_date}' AND cashflows_date <= '{valuation_date}'
    """, conn)

print('Data loaded.')

In [ ]:
# ── Pivot metrics → wide, then split prev / curr ──────────────────────────────
df_m_wide = df_m.pivot_table(index=['RepDate','isin'], columns='metric', values='value').reset_index()
df_m_wide.columns.name = None

m_prev = df_m_wide[df_m_wide.RepDate == prev_date].drop('RepDate', axis=1)
m_curr = df_m_wide[df_m_wide.RepDate == valuation_date].drop('RepDate', axis=1)

p_prev = df_p[df_p.RepDate == prev_date][['isin','oas_spread']].rename(columns={'oas_spread':'oas_prev'})
p_curr = df_p[df_p.RepDate == valuation_date][['isin','oas_spread','pricing_source']].rename(columns={'oas_spread':'oas_curr'})

# Aggregate payments per ISIN
if not df_cf.empty:
    cf_agg = df_cf.groupby('isin').agg(
        coupon_paid=('coupon','sum'),
        principal_paid=('principal','sum'),
        payment_dates=('cashflows_date', lambda x: ', '.join(x.astype(str)))
    ).reset_index()
else:
    cf_agg = pd.DataFrame(columns=['isin','coupon_paid','principal_paid','payment_dates'])

# ── Merge everything into one table ──────────────────────────────────────────
df = m_prev.merge(m_curr, on='isin', suffixes=('_prev','_curr'), how='inner')
df = df.merge(p_prev, on='isin', how='left')
df = df.merge(p_curr, on='isin', how='left')
df = df.merge(df_bonds, on='isin', how='left')
df = df.merge(df_pos[['isin','portfolio_type','currency','quantity','face_value']], on='isin', how='left')
df = df.merge(cf_agg, on='isin', how='left')
df['coupon_paid']    = df['coupon_paid'].fillna(0)
df['principal_paid'] = df['principal_paid'].fillna(0)
df['payment_dates']  = df['payment_dates'].fillna('')

print(f'Merged table: {df.shape[0]} bonds')

In [ ]:
# ── Calculate attribution components ─────────────────────────────────────────
#
#  rate_effect   = -dv01 × Δrf_bps       (dv01 in price% per 1bp)
#  spread_effect = -cr01 × Δoas_bps      (cr01 in price% per 1bp)
#  pull_to_par   = (100 - price_prev) / dtm_prev   (daily drift toward 100)
#  residual      = total_change - rate - spread - pull

df['price_prev']   = df['clean_price_hybrid_prev']
df['price_curr']   = df['clean_price_hybrid_curr']
df['price_change'] = (df['price_curr'] - df['price_prev']).round(4)

df['delta_rf_bps']  = (df['par_yield_rf_rate_pp_curr'] - df['par_yield_rf_rate_pp_prev']) * 100
df['delta_oas_bps'] = df['oas_curr'] - df['oas_prev']

df['rate_effect']   = (-df['dv01_prev']  * df['delta_rf_bps']).fillna(0).round(4)
df['spread_effect'] = (-df['cr01_prev']  * df['delta_oas_bps']).fillna(0).round(4)
df['pull_to_par']   = np.where(df['dtm_prev'] > 0,
                               (100 - df['price_prev']) / df['dtm_prev'], 0).round(4)
df['residual']      = (df['price_change']
                       - df['rate_effect']
                       - df['spread_effect']
                       - df['pull_to_par']).round(4)

print('Attribution calculated.')

In [ ]:
# ── Build human-readable "reason" text ───────────────────────────────────────
#
# For each bond, explains what drove the price change in plain text.
# Only mentions components that are >= 0.05 in absolute value.

MIN_COMPONENT = 0.05   # ignore tiny components in the explanation

def build_reason(row):
    parts = []

    # Rate effect
    if abs(row.get('rate_effect', 0)) >= MIN_COMPONENT:
        rf_dir  = 'rose' if row['delta_rf_bps'] > 0 else 'fell'
        parts.append(
            f"risk-free rate {rf_dir} {abs(row['delta_rf_bps']):.1f}bps "
            f"→ price {row['rate_effect']:+.4f}"
        )

    # Spread effect
    if abs(row.get('spread_effect', 0)) >= MIN_COMPONENT:
        sp_dir = 'widened' if row['delta_oas_bps'] > 0 else 'tightened'
        parts.append(
            f"credit spread {sp_dir} {abs(row['delta_oas_bps']):.1f}bps "
            f"→ price {row['spread_effect']:+.4f}"
        )

    # Pull to par
    if abs(row.get('pull_to_par', 0)) >= MIN_COMPONENT:
        direction = 'up' if row['pull_to_par'] > 0 else 'down'
        parts.append(
            f"pull to par (converging {direction}) "
            f"→ price {row['pull_to_par']:+.4f}"
        )

    # Coupon / principal payment
    if row.get('coupon_paid', 0) > 0:
        parts.append(f"coupon paid {row['coupon_paid']:.4f} on {row['payment_dates']}")
    if row.get('principal_paid', 0) > 0:
        parts.append(f"principal repaid {row['principal_paid']:.4f} on {row['payment_dates']}")

    # Residual
    if abs(row.get('residual', 0)) >= MIN_COMPONENT:
        parts.append(f"other/unexplained {row['residual']:+.4f}")

    return '  |  '.join(parts) if parts else 'No significant driver identified'

df['reason'] = df.apply(build_reason, axis=1)
print('Reasons built.')

In [ ]:
# ── Filter: only bonds with |price_change| > threshold ───────────────────────
df_significant = df[df['price_change'].abs() > threshold].copy()
df_significant = df_significant.sort_values('price_change')

total_bonds      = df.shape[0]
significant_bonds = df_significant.shape[0]
stable_bonds     = total_bonds - significant_bonds

print(f'Total bonds in portfolio : {total_bonds}')
print(f'Price change > {threshold}  : {significant_bonds} bonds  ← shown in report')
print(f'Price stable (≤ {threshold}) : {stable_bonds} bonds')

In [ ]:
# ── Readable report table (HTML display in notebook) ─────────────────────────

def color_price_change(val):
    """Red for negative, green for positive price change."""
    color = '#4ade80' if val > 0 else '#f87171'
    return f'color: {color}; font-weight: bold'

def color_component(val):
    """Light red/green for attribution components."""
    if pd.isna(val) or val == 0:
        return ''
    return 'color: #86efac' if val > 0 else 'color: #fca5a5'

report_cols = {
    'issuer'        : 'Issuer',
    'isin'          : 'ISIN',
    'portfolio_type': 'Portfolio',
    'currency'      : 'CCY',
    'price_prev'    : f'Price\n{prev_date}',
    'price_curr'    : f'Price\n{valuation_date}',
    'price_change'  : 'Change',
    'rate_effect'   : 'Rate\nEffect',
    'spread_effect' : 'Spread\nEffect',
    'pull_to_par'   : 'Pull\nto Par',
    'residual'      : 'Residual',
    'reason'        : 'Explanation'
}

df_report = df_significant[list(report_cols.keys())].rename(columns=report_cols).copy()

styled = (
    df_report.style
    .applymap(color_price_change, subset=['Change'])
    .applymap(color_component, subset=['Rate\nEffect', 'Spread\nEffect', 'Pull\nto Par', 'Residual'])
    .format({
        f'Price\n{prev_date}'  : '{:.4f}',
        f'Price\n{valuation_date}': '{:.4f}',
        'Change'               : '{:+.4f}',
        'Rate\nEffect'         : '{:+.4f}',
        'Spread\nEffect'       : '{:+.4f}',
        'Pull\nto Par'         : '{:+.4f}',
        'Residual'             : '{:+.4f}',
    }, na_rep='—')
    .set_table_styles([
        {'selector': 'table',
         'props': [('background-color','#111827'), ('color','#e2e8f0'),
                   ('border-collapse','collapse'), ('font-family','Arial'),
                   ('font-size','13px'), ('width','100%')]},
        {'selector': 'th',
         'props': [('background-color','#1e2534'), ('color','#94a3b8'),
                   ('padding','8px 12px'), ('text-align','center'),
                   ('border','1px solid #1e2534'), ('white-space','pre-line')]},
        {'selector': 'td',
         'props': [('padding','7px 12px'), ('border','1px solid #1e2534'),
                   ('vertical-align','middle')]},
        {'selector': 'tr:hover td',
         'props': [('background-color','#1e2534')]},
    ])
    .set_properties(**{'text-align': 'left'})
    .hide(axis='index')
)

# Header banner
header_html = f"""
<div style="background:#0a0e1a; padding:16px 20px; border-radius:8px;
            border-left:4px solid #00b4d8; margin-bottom:16px; font-family:Arial">
  <div style="color:#00b4d8; font-size:18px; font-weight:bold">
    Bond Price Attribution Report
  </div>
  <div style="color:#94a3b8; font-size:13px; margin-top:4px">
    Period: <b style="color:#e2e8f0">{prev_date}</b> → <b style="color:#e2e8f0">{valuation_date}</b>
    &nbsp;&nbsp;|&nbsp;&nbsp;
    Threshold: <b style="color:#e2e8f0">|Δ| &gt; {threshold}</b>
    &nbsp;&nbsp;|&nbsp;&nbsp;
    Showing <b style="color:#e2e8f0">{significant_bonds}</b> of <b style="color:#e2e8f0">{total_bonds}</b> bonds
    &nbsp;&nbsp;|&nbsp;&nbsp;
    <span style="color:#94a3b8">{stable_bonds} bonds stable (not shown)</span>
  </div>
</div>
"""

display(HTML(header_html))

if df_significant.empty:
    display(HTML('<div style="color:#4ade80; font-family:Arial; padding:12px">'
                 'All bonds are stable today — no price change exceeds the threshold.</div>'))
else:
    display(styled)

In [ ]:
# ── Portfolio-level summary ───────────────────────────────────────────────────
summary = df.groupby('portfolio_type').agg(
    bonds=('isin','count'),
    avg_price_change=('price_change','mean'),
    avg_rate_effect=('rate_effect','mean'),
    avg_spread_effect=('spread_effect','mean'),
    avg_pull_to_par=('pull_to_par','mean'),
    avg_residual=('residual','mean'),
    bonds_moved=('price_change', lambda x: (x.abs() > threshold).sum())
).round(4).reset_index()

summary_html = f"""
<div style="background:#0a0e1a; padding:16px 20px; border-radius:8px;
            border-left:4px solid #4ade80; margin:16px 0; font-family:Arial">
  <div style="color:#4ade80; font-size:15px; font-weight:bold; margin-bottom:10px">
    Portfolio Summary
  </div>
"""

for _, row in summary.iterrows():
    chg_color = '#4ade80' if row['avg_price_change'] >= 0 else '#f87171'
    summary_html += f"""
  <div style="background:#111827; border-radius:6px; padding:10px 14px; margin-bottom:8px;
              border:1px solid #1e2534">
    <span style="color:#e2e8f0; font-weight:bold; font-size:14px">{row['portfolio_type']}</span>
    &nbsp;&nbsp;
    <span style="color:#94a3b8; font-size:12px">{int(row['bonds'])} bonds,
      {int(row['bonds_moved'])} moved &gt;{threshold}</span>
    <br/>
    <span style="color:#94a3b8; font-size:12px">
      Avg change: <b style="color:{chg_color}">{row['avg_price_change']:+.4f}</b>
      &nbsp;|&nbsp;
      Rate effect: <b style="color:#94a3b8">{row['avg_rate_effect']:+.4f}</b>
      &nbsp;|&nbsp;
      Spread effect: <b style="color:#94a3b8">{row['avg_spread_effect']:+.4f}</b>
      &nbsp;|&nbsp;
      Pull to par: <b style="color:#94a3b8">{row['avg_pull_to_par']:+.4f}</b>
      &nbsp;|&nbsp;
      Residual: <b style="color:#94a3b8">{row['avg_residual']:+.4f}</b>
    </span>
  </div>
"""

summary_html += '</div>'
display(HTML(summary_html))

In [ ]:
# ── Chart: attribution breakdown for significant bonds ────────────────────────
if not df_significant.empty:
    components = ['rate_effect', 'spread_effect', 'pull_to_par', 'residual']
    labels     = ['Rate Effect', 'Spread Effect', 'Pull to Par', 'Residual']
    colors     = ['#ef4444', '#f97316', '#4ade80', '#94a3b8']

    fig = go.Figure()

    for comp, label, color in zip(components, labels, colors):
        fig.add_trace(go.Bar(
            name=label,
            x=df_significant['isin'],
            y=df_significant[comp],
            marker_color=color,
            hovertemplate=(
                '<b>%{x}</b><br>'
                + label + ': %{y:+.4f}<extra></extra>'
            )
        ))

    # Diamond marker for total price change
    fig.add_trace(go.Scatter(
        name='Total Change',
        x=df_significant['isin'],
        y=df_significant['price_change'],
        mode='markers',
        marker=dict(color='#00b4d8', size=9, symbol='diamond'),
        hovertemplate='<b>%{x}</b><br>Total: %{y:+.4f}<extra></extra>'
    ))

    fig.update_layout(
        title=dict(
            text=f'Price Change Attribution  |  {prev_date} → {valuation_date}  '
                 f'(bonds with |Δ| > {threshold})',
            font=dict(size=15, color='#e2e8f0')
        ),
        barmode='stack',
        template='plotly_dark',
        paper_bgcolor='#0a0e1a',
        plot_bgcolor='#111827',
        xaxis=dict(title='ISIN', tickangle=-45, tickfont=dict(color='#94a3b8', size=10)),
        yaxis=dict(title='Price Change (% of par)', tickfont=dict(color='#94a3b8'),
                   gridcolor='#1e2534', zeroline=True, zerolinecolor='#475569'),
        legend=dict(font=dict(color='#94a3b8'), bgcolor='#111827', bordercolor='#1e2534'),
        margin=dict(l=60, r=40, t=80, b=120),
        height=520
    )
    fig.show()

In [ ]:
# ── Save to Excel ─────────────────────────────────────────────────────────────
import os, openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

# Sheet 1: Significant movers
excel_cols_sig = [
    'issuer','isin','portfolio_type','currency',
    'price_prev','price_curr','price_change',
    'rate_effect','spread_effect','pull_to_par','residual',
    'delta_rf_bps','delta_oas_bps',
    'mod_duration_prev','dv01_prev','cr01_prev','dtm_prev',
    'coupon_paid','principal_paid','payment_dates',
    'reason'
]
excel_cols_sig = [c for c in excel_cols_sig if c in df_significant.columns]

df_sig_out = df_significant[excel_cols_sig].rename(columns={
    'price_prev'        : f'Price {prev_date}',
    'price_curr'        : f'Price {valuation_date}',
    'price_change'      : 'Price Change',
    'rate_effect'       : 'Rate Effect',
    'spread_effect'     : 'Spread Effect',
    'pull_to_par'       : 'Pull to Par',
    'residual'          : 'Residual',
    'delta_rf_bps'      : 'Δ RF Rate (bps)',
    'delta_oas_bps'     : 'Δ OAS (bps)',
    'mod_duration_prev' : 'Mod Duration',
    'dv01_prev'         : 'DV01',
    'cr01_prev'         : 'CR01',
    'dtm_prev'          : 'Days to Mat.',
    'coupon_paid'       : 'Coupon Paid',
    'principal_paid'    : 'Principal Paid',
    'payment_dates'     : 'Payment Dates',
    'reason'            : 'Explanation'
})

# Sheet 2: All bonds
all_cols = ['issuer','isin','portfolio_type','currency',
            'price_prev','price_curr','price_change',
            'rate_effect','spread_effect','pull_to_par','residual','reason']
all_cols = [c for c in all_cols if c in df.columns]
df_all_out = df[all_cols].sort_values(['portfolio_type','price_change']).rename(columns={
    'price_prev'   : f'Price {prev_date}',
    'price_curr'   : f'Price {valuation_date}',
    'price_change' : 'Price Change',
    'rate_effect'  : 'Rate Effect',
    'spread_effect': 'Spread Effect',
    'pull_to_par'  : 'Pull to Par',
    'residual'     : 'Residual',
    'reason'       : 'Explanation'
})

with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    df_sig_out.to_excel(writer, sheet_name='Significant Changes', index=False)
    df_all_out.to_excel(writer, sheet_name='All Bonds', index=False)
    summary.to_excel(writer, sheet_name='Portfolio Summary', index=False)

    # ── Basic formatting for 'Significant Changes' sheet ──────────────────
    ws = writer.sheets['Significant Changes']
    header_fill = PatternFill('solid', fgColor='1E2534')
    header_font = Font(color='94A3B8', bold=True, size=10)
    green_font  = Font(color='16A34A', bold=True)
    red_font    = Font(color='DC2626', bold=True)
    center_align = Alignment(horizontal='center', vertical='center', wrap_text=False)
    wrap_align   = Alignment(horizontal='left',   vertical='top',    wrap_text=True)

    for cell in ws[1]:  # header row
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = center_align

    price_change_col = None
    for i, col in enumerate(df_sig_out.columns, 1):
        if col == 'Price Change':
            price_change_col = i
        if col == 'Explanation':
            ws.column_dimensions[get_column_letter(i)].width = 60
        elif col in ('issuer',):
            ws.column_dimensions[get_column_letter(i)].width = 22
        elif col in ('isin',):
            ws.column_dimensions[get_column_letter(i)].width = 14
        else:
            ws.column_dimensions[get_column_letter(i)].width = 13

    if price_change_col:
        for row in ws.iter_rows(min_row=2, min_col=price_change_col, max_col=price_change_col):
            for cell in row:
                if cell.value is not None:
                    cell.font = green_font if cell.value > 0 else red_font

    ws.freeze_panes = 'A2'
    ws.row_dimensions[1].height = 22

print(f'Saved: {output_file_path}')